## StatefulSets & `volumeClaimTemplates` — per-pod PVCs

A Deployment shares one pod template across replicas — so a `persistentVolumeClaim` in that template means every replica mounts the **same** PVC, usually `RWO` on one node. You can't scale past one. A **StatefulSet** fixes this with a **`volumeClaimTemplate`**:

```yaml
apiVersion: apps/v1
kind: StatefulSet
metadata: { name: cassandra }
spec:
  serviceName: cassandra          # headless Service (notebook 04)
  replicas: 3
  selector: { matchLabels: { app: cassandra } }
  template:
    metadata: { labels: { app: cassandra } }
    spec:
      containers:
        - name: app
          image: cassandra:5
          volumeMounts: [{ name: data, mountPath: /var/lib/cassandra }]
  volumeClaimTemplates:
    - metadata: { name: data }
      spec:
        accessModes: [ReadWriteOnce]
        resources: { requests: { storage: 10Gi } }
```

The controller creates a PVC **per replica** — `data-cassandra-0`, `-1`, `-2` — each bound to its own PV, each mounted by its pod. Restart or reschedule a pod and **its PVC stays**: the new `cassandra-0` remounts `data-cassandra-0`. Stable identity, stable storage.

**Deletion is deliberate:** deleting the StatefulSet **does not** delete the PVCs (your data is precious). Clean up manually with `kubectl delete pvc -l app=cassandra`. 1.27+ adds `persistentVolumeClaimRetentionPolicy` to opt into cascade — choose deliberately.

This is why every database in Kubernetes uses StatefulSets: stable **name** (`cassandra-0`), stable **network identity** (`cassandra-0.cassandra...` via the headless Service), stable **storage** (`data-cassandra-0`). On our map this section links the **StatefulSet** (Workload kinds) to a per-pod **PVC** (Storage) — the triplet a database replica needs.